In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 1. Upload and load dataset
uploaded = files.upload()
filename = next(iter(uploaded))
data = pd.read_csv(filename)

print("DATASET PREVIEW")
print(data.head())
print("\nDataset Shape:", data.shape)
print("\nMissing Values:\n", data.isnull().sum())

# 2. Prepare dataset
target = "Yield_ton"

if target not in data.columns:
    raise ValueError(
        f"Target column '{target}' not found. "
        f"Available columns: {list(data.columns)}"
    )

X = data.drop(columns=[target])
y = data[target]

# Convert categorical features into numeric columns
X = pd.get_dummies(X, drop_first=True)

# Handle missing and infinite values
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

y = y.replace([np.inf, -np.inf], np.nan)
valid_rows = y.notna()
X = X.loc[valid_rows]
y = y.loc[valid_rows]

if len(X) < 5:
    raise ValueError("At least 5 valid rows are needed for this case study.")

# 3. Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("\nTraining Records:", len(X_train))
print("Testing Records:", len(X_test))

# 4. Create machine learning models
models = {
    "Linear Regression": LinearRegression(),
    "Shallow Tree": DecisionTreeRegressor(
        max_depth=2, random_state=42
    ),
    "Deep Tree": DecisionTreeRegressor(
        max_depth=None, random_state=42
    )
}

results = {}

# 5. Train and evaluate all models
for name, model in models.items():
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    results[name] = {
        "model": model,
        "train_pred": train_pred,
        "test_pred": test_pred,
        "train_r2": r2_score(y_train, train_pred),
        "test_r2": r2_score(y_test, test_pred),
        "train_mse": mean_squared_error(y_train, train_pred),
        "test_mse": mean_squared_error(y_test, test_pred),
        "test_mae": mean_absolute_error(y_test, test_pred)
    }

# 6. Display model comparison
print("\nMODEL PERFORMANCE COMPARISON")

for name, result in results.items():
    print("\nModel:", name)
    print("Training R2:", round(result["train_r2"], 4))
    print("Testing R2:", round(result["test_r2"], 4))
    print("Training MSE:", round(result["train_mse"], 4))
    print("Testing MSE:", round(result["test_mse"], 4))
    print("Testing MAE:", round(result["test_mae"], 4))

# 7. Graph 1: Training vs Testing R2
names = list(results.keys())
train_scores = [results[n]["train_r2"] for n in names]
test_scores = [results[n]["test_r2"] for n in names]

x = np.arange(len(names))
width = 0.35

plt.figure(figsize=(9, 5))
plt.bar(x - width / 2, train_scores, width, label="Training R2")
plt.bar(x + width / 2, test_scores, width, label="Testing R2")
plt.xticks(x, names, rotation=15)
plt.ylabel("R2 Score")
plt.title("Training vs Testing Performance")
plt.legend()
plt.tight_layout()
plt.show()

# 8. Graph 2: Training vs Testing MSE
train_mse = [results[n]["train_mse"] for n in names]
test_mse = [results[n]["test_mse"] for n in names]

plt.figure(figsize=(9, 5))
plt.bar(x - width / 2, train_mse, width, label="Training MSE")
plt.bar(x + width / 2, test_mse, width, label="Testing MSE")
plt.xticks(x, names, rotation=15)
plt.ylabel("Mean Squared Error")
plt.title("Training vs Testing Error")
plt.legend()
plt.tight_layout()
plt.show()

# 9. Graph 3: Actual vs Predicted Yield
best_name = max(names, key=lambda n: results[n]["test_r2"])
best_model = results[best_name]["model"]
best_pred = results[best_name]["test_pred"]

plt.figure(figsize=(7, 5))
plt.scatter(y_test, best_pred)

low = min(y_test.min(), best_pred.min())
high = max(y_test.max(), best_pred.max())

plt.plot([low, high], [low, high], linestyle="--")
plt.xlabel("Actual Yield")
plt.ylabel("Predicted Yield")
plt.title("Actual vs Predicted Yield - " + best_name)
plt.tight_layout()
plt.show()

print("\nModel with highest test R2 in this split:", best_name)

# 10. Graph 4: Prediction Error / Residuals
residuals = y_test.to_numpy() - best_pred

plt.figure(figsize=(8, 5))
plt.scatter(best_pred, residuals)
plt.axhline(y=0, linestyle="--")
plt.xlabel("Predicted Yield")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Prediction Error Plot - " + best_name)
plt.tight_layout()
plt.show()

# 11. Graph 5: Residual Distribution
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=10, edgecolor="black")
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Distribution of Prediction Errors")
plt.tight_layout()
plt.show()

# 12. Graph 6: Model Complexity and Overfitting
depths = [1, 2, 3, 5, 10, None]
depth_labels = []
depth_train_r2 = []
depth_test_r2 = []

for depth in depths:
    tree = DecisionTreeRegressor(
        max_depth=depth, random_state=42
    )
    tree.fit(X_train, y_train)

    depth_train_r2.append(
        r2_score(y_train, tree.predict(X_train))
    )
    depth_test_r2.append(
        r2_score(y_test, tree.predict(X_test))
    )

    depth_labels.append(
        "Unlimited" if depth is None else str(depth)
    )

plt.figure(figsize=(9, 5))
plt.plot(
    depth_labels, depth_train_r2,
    marker="o", label="Training R2"
)
plt.plot(
    depth_labels, depth_test_r2,
    marker="o", label="Testing R2"
)
plt.xlabel("Maximum Tree Depth")
plt.ylabel("R2 Score")
plt.title("Effect of Model Complexity")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# 13. K-Fold Cross-Validation
n_splits = min(5, len(X))
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

print("\nK-FOLD CROSS-VALIDATION RESULTS")

cv_scores = cross_val_score(
    LinearRegression(),
    X, y,
    cv=kf,
    scoring="r2"
)

print("Fold Scores:", np.round(cv_scores, 4))
print("Average R2 Score:", round(cv_scores.mean(), 4))
print("Standard Deviation:", round(cv_scores.std(), 4))

# 14. Graph 7: K-Fold Scores
plt.figure(figsize=(8, 5))
plt.bar(
    [f"Fold {i+1}" for i in range(len(cv_scores))],
    cv_scores,
    edgecolor="black"
)
plt.axhline(
    y=cv_scores.mean(),
    linestyle="--",
    label="Average R2"
)
plt.xlabel("Cross-Validation Fold")
plt.ylabel("R2 Score")
plt.title("K-Fold Cross-Validation Results")
plt.legend()
plt.tight_layout()
plt.show()

# 15. Graph 8: Linear Regression Coefficients
linear_model = results["Linear Regression"]["model"]

coefficients = pd.Series(
    linear_model.coef_,
    index=X.columns
).sort_values()

plt.figure(figsize=(9, 5))
coefficients.plot(kind="barh")
plt.xlabel("Coefficient Value")
plt.ylabel("Input Feature")
plt.title("Linear Regression Feature Coefficients")
plt.tight_layout()
plt.show()

print("\nCASE STUDY COMPLETED SUCCESSFULLY")
